# Notebook 01 — AdapTime Reproduction Scaffold

**Repo:** `temporal-phase-lock`  
**Notebook:** `notebooks/01_adaptime_reproduction.ipynb`  
**Purpose:** reproduce AdapTime-style adaptive temporal reasoning as a transparent baseline before adding constraint-gated phase-lock validation.

This notebook keeps the same working pattern used across recent repo notebooks:

- self-contained imports and deterministic seed
- compact synthetic dataset
- reusable pipeline functions
- baseline metrics table
- publication-style figures
- saved artifacts for `docs/` and `figures/`
- clear next-notebook handoff

## Core baseline loop

```text
question
  → reformulate
  → rewrite temporal evidence
  → review answer against evidence
  → final answer + trace
```

This notebook does **not** claim to reproduce private training or hidden implementation details from any paper.  
It implements an open, inspectable reproduction scaffold of the *reasoning pattern*.

## 0. Environment setup

In [ ]:
import os
import re
import math
import json
import random
import textwrap
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 9423
random.seed(SEED)
np.random.seed(SEED)

REPO_NAME = "temporal-phase-lock"
NOTEBOOK_ID = "01_adaptime_reproduction"

BASE_DIR = Path(".")
FIG_DIR = BASE_DIR / "figures"
DOCS_DIR = BASE_DIR / "docs"
ARTIFACT_DIR = BASE_DIR / "artifacts"

for d in [FIG_DIR, DOCS_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Ready: {REPO_NAME}/{NOTEBOOK_ID}")
print(f"Seed: {SEED}")

## 1. Synthetic temporal QA set

We start with small hand-checkable examples.  
Each case includes temporal evidence, a question, a gold answer, and a lightweight type label.

The point is not dataset scale yet.  
The point is **trace visibility**: can we see each reasoning step?

In [ ]:
@dataclass
class TemporalCase:
    case_id: str
    question: str
    evidence: List[str]
    gold: str
    answer_type: str

CASES = [
    TemporalCase(
        case_id="TQA-001",
        question="Who joined after Mira but before Theo?",
        evidence=[
            "Mira joined the project in March.",
            "Jon joined the project in May.",
            "Theo joined the project in August.",
            "Leah joined the project in November."
        ],
        gold="Jon",
        answer_type="between"
    ),
    TemporalCase(
        case_id="TQA-002",
        question="Which event happened first?",
        evidence=[
            "The calibration finished on Tuesday.",
            "The sensor test started on Monday.",
            "The report was drafted on Wednesday."
        ],
        gold="sensor test",
        answer_type="first"
    ),
    TemporalCase(
        case_id="TQA-003",
        question="What happened immediately after the review?",
        evidence=[
            "The draft happened before the review.",
            "The review happened before the revision.",
            "The revision happened before the release."
        ],
        gold="revision",
        answer_type="after"
    ),
    TemporalCase(
        case_id="TQA-004",
        question="Which item was completed last?",
        evidence=[
            "Notebook 01 was completed before Notebook 02.",
            "Notebook 03 was completed after Notebook 02.",
            "Notebook 04 was completed after Notebook 03."
        ],
        gold="Notebook 04",
        answer_type="last"
    ),
    TemporalCase(
        case_id="TQA-005",
        question="Did the audit occur before deployment?",
        evidence=[
            "The prototype was tested in April.",
            "The audit occurred in June.",
            "Deployment occurred in September."
        ],
        gold="Yes",
        answer_type="yes_no"
    ),
]

case_df = pd.DataFrame([asdict(c) for c in CASES])
case_df

## 2. Temporal normalization utilities

A simple reproduction baseline needs explicit temporal parsing.

This first pass supports:

- calendar months
- weekdays
- ordinal event ordering from `before` and `after`
- yes/no temporal comparison

In [ ]:
MONTH_ORDER = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12
}

WEEKDAY_ORDER = {
    "monday": 1, "tuesday": 2, "wednesday": 3, "thursday": 4,
    "friday": 5, "saturday": 6, "sunday": 7
}

def clean_phrase(text: str) -> str:
    text = re.sub(r"\.$", "", text.strip())
    text = re.sub(r"^(the|a|an)\s+", "", text, flags=re.I)
    return text

def extract_subject(sentence: str) -> str:
    parts = re.split(
        r"\s+(joined|occurred|finished|started|was|were|completed|happened|drafted|tested)\b",
        sentence,
        maxsplit=1,
        flags=re.I,
    )
    return clean_phrase(parts[0])

def extract_month_fact(sentence: str) -> Optional[Tuple[str, int]]:
    s = sentence.lower()
    for month, idx in MONTH_ORDER.items():
        if month in s:
            return extract_subject(sentence), idx
    return None

def extract_weekday_fact(sentence: str) -> Optional[Tuple[str, int]]:
    s = sentence.lower()
    for day, idx in WEEKDAY_ORDER.items():
        if day in s:
            return extract_subject(sentence), idx
    return None

def extract_before_after(sentence: str) -> List[Tuple[str, str, str]]:
    s = sentence.strip().rstrip(".")
    out = []

    m = re.match(r"(.+?)\s+was completed before\s+(.+)", s, flags=re.I)
    if m:
        out.append((clean_phrase(m.group(1)), "before", clean_phrase(m.group(2))))
        return out

    m = re.match(r"(.+?)\s+was completed after\s+(.+)", s, flags=re.I)
    if m:
        out.append((clean_phrase(m.group(2)), "before", clean_phrase(m.group(1))))
        return out

    m = re.match(r"(.+?)\s+happened before\s+(.+)", s, flags=re.I)
    if m:
        out.append((clean_phrase(m.group(1)), "before", clean_phrase(m.group(2))))
        return out

    return out

def normalize_evidence(evidence: List[str]) -> Dict:
    facts = {}
    constraints = []

    for sent in evidence:
        mf = extract_month_fact(sent)
        wf = extract_weekday_fact(sent)
        if mf:
            facts[mf[0]] = mf[1]
        if wf:
            facts[mf[0] if False else wf[0]] = wf[1]
        constraints.extend(extract_before_after(sent))

    return {"facts": facts, "constraints": constraints}

for c in CASES:
    print(c.case_id, normalize_evidence(c.evidence))

## 3. Baseline AdapTime-style stages

We implement three transparent stages:

### A. Reformulate
Turn question into a small structured intent.

### B. Rewrite
Turn evidence into temporal facts and constraints.

### C. Review
Generate answer and check whether evidence supports it.

This makes later notebooks easy to upgrade with phase-lock scoring.

In [ ]:
def reformulate(question: str) -> Dict:
    q = question.lower()
    if "after" in q and "before" in q:
        return {"intent": "between", "target": "entity_between_two_anchors"}
    if "happened first" in q or "event happened first" in q:
        return {"intent": "first", "target": "earliest_event"}
    if "immediately after" in q:
        return {"intent": "after", "target": "successor_event"}
    if "completed last" in q or " last" in q:
        return {"intent": "last", "target": "latest_event"}
    if q.startswith("did") and "before" in q:
        return {"intent": "yes_no_before", "target": "temporal_comparison"}
    return {"intent": "unknown", "target": "unknown"}

def topological_order_from_constraints(constraints: List[Tuple[str, str, str]]) -> List[str]:
    nodes = sorted(set([a for a, _, b in constraints] + [b for a, _, b in constraints]))
    incoming = {n: set() for n in nodes}
    outgoing = {n: set() for n in nodes}
    for a, rel, b in constraints:
        if rel == "before":
            outgoing[a].add(b)
            incoming[b].add(a)
    order = []
    ready = sorted([n for n in nodes if not incoming[n]])
    while ready:
        n = ready.pop(0)
        order.append(n)
        for m in sorted(outgoing[n]):
            incoming[m].discard(n)
            if not incoming[m]:
                ready.append(m)
        ready = sorted(set(ready))
    return order

def rewrite_temporal_evidence(evidence: List[str]) -> Dict:
    normalized = normalize_evidence(evidence)
    facts = normalized["facts"]
    constraints = normalized["constraints"]

    timeline = []
    if facts:
        timeline = [k for k, v in sorted(facts.items(), key=lambda kv: kv[1])]
    elif constraints:
        timeline = topological_order_from_constraints(constraints)

    return {
        "facts": facts,
        "constraints": constraints,
        "timeline": timeline
    }

def answer_from_trace(question: str, reformulated: Dict, rewritten: Dict) -> str:
    intent = reformulated["intent"]
    timeline = rewritten["timeline"]
    facts = rewritten["facts"]

    if intent == "between":
        m = re.search(r"after\s+([A-Za-z0-9 ]+?)\s+but\s+before\s+([A-Za-z0-9 ]+)\?", question, flags=re.I)
        if not m:
            return "UNKNOWN"
        left = clean_phrase(m.group(1))
        right = clean_phrase(m.group(2))
        if left in facts and right in facts:
            between = [name for name, t in facts.items() if facts[left] < t < facts[right]]
            return between[0] if between else "UNKNOWN"
        return "UNKNOWN"

    if intent == "first":
        return timeline[0] if timeline else "UNKNOWN"

    if intent == "after":
        anchor = "review"
        matches = [i for i, item in enumerate(timeline) if anchor.lower() in item.lower()]
        if matches and matches[0] + 1 < len(timeline):
            return timeline[matches[0] + 1]
        return "UNKNOWN"

    if intent == "last":
        return timeline[-1] if timeline else "UNKNOWN"

    if intent == "yes_no_before":
        m = re.match(r"did\s+(.+?)\s+occur\s+before\s+(.+?)\?", question, flags=re.I)
        if m:
            a = clean_phrase(m.group(1))
            b = clean_phrase(m.group(2))
            a_key = next((k for k in facts if a.lower() in k.lower() or k.lower() in a.lower()), None)
            b_key = next((k for k in facts if b.lower() in k.lower() or k.lower() in b.lower()), None)
            if a_key and b_key:
                return "Yes" if facts[a_key] < facts[b_key] else "No"
        return "UNKNOWN"

    return "UNKNOWN"

def review_answer(pred: str, gold: str, rewritten: Dict) -> Dict:
    pred_norm = pred.lower().strip()
    gold_norm = gold.lower().strip()
    exact = pred_norm == gold_norm or pred_norm in gold_norm or gold_norm in pred_norm
    support = pred != "UNKNOWN" and (pred in rewritten["timeline"] or pred in rewritten["facts"] or pred in ["Yes", "No"])
    return {
        "exact_match": bool(exact),
        "supported": bool(support),
        "review_status": "PASS" if exact and support else "CHECK"
    }

def run_adaptime_style_case(case: TemporalCase) -> Dict:
    r = reformulate(case.question)
    w = rewrite_temporal_evidence(case.evidence)
    pred = answer_from_trace(case.question, r, w)
    review = review_answer(pred, case.gold, w)
    return {
        "case_id": case.case_id,
        "question": case.question,
        "gold": case.gold,
        "prediction": pred,
        "answer_type": case.answer_type,
        "intent": r["intent"],
        "timeline": " → ".join(w["timeline"]),
        "num_evidence": len(case.evidence),
        **review
    }

results = pd.DataFrame([run_adaptime_style_case(c) for c in CASES])
results

## 4. Baseline metrics

In [ ]:
accuracy = results["exact_match"].mean()
support_rate = results["supported"].mean()
pass_rate = (results["review_status"] == "PASS").mean()

metrics = pd.DataFrame([
    {"metric": "exact_match_accuracy", "value": accuracy},
    {"metric": "support_rate", "value": support_rate},
    {"metric": "review_pass_rate", "value": pass_rate},
    {"metric": "num_cases", "value": len(results)},
])

metrics

## 5. Trace table

In [ ]:
trace_cols = [
    "case_id", "answer_type", "intent", "question",
    "timeline", "prediction", "gold", "supported", "review_status"
]
trace_table = results[trace_cols].copy()
trace_table

## 6. Figure 1 — baseline review outcomes

In [ ]:
status_counts = results["review_status"].value_counts().reindex(["PASS", "CHECK"], fill_value=0)

plt.figure(figsize=(7, 4.5))
plt.bar(status_counts.index, status_counts.values)
plt.title("AdapTime-style baseline review outcomes")
plt.xlabel("Review status")
plt.ylabel("Number of cases")
plt.tight_layout()

fig1_path = FIG_DIR / "01_baseline_review_outcomes.png"
plt.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 7. Figure 2 — stage-level scaffold

In [ ]:
stages = ["Question", "Reformulate", "Rewrite", "Review", "Answer"]
x = np.arange(len(stages))
y = np.zeros(len(stages))

plt.figure(figsize=(10, 2.8))
plt.plot(x, y, marker="o", linewidth=2)
for i, label in enumerate(stages):
    plt.text(i, 0.08, label, ha="center", va="bottom", fontsize=11)
    if i < len(stages) - 1:
        plt.text(i + 0.5, -0.09, "→", ha="center", va="top", fontsize=16)
plt.ylim(-0.25, 0.35)
plt.yticks([])
plt.xticks([])
plt.title("Adaptive temporal reasoning scaffold")
plt.tight_layout()

fig2_path = FIG_DIR / "01_adaptime_style_pipeline.png"
plt.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

## 8. Figure 3 — temporal trace lengths

In [ ]:
timeline_lengths = []
for c in CASES:
    rewritten = rewrite_temporal_evidence(c.evidence)
    timeline_lengths.append(len(rewritten["timeline"]))

plt.figure(figsize=(8, 4.5))
plt.bar([c.case_id for c in CASES], timeline_lengths)
plt.title("Recovered temporal trace length by case")
plt.xlabel("Case")
plt.ylabel("Timeline nodes")
plt.xticks(rotation=20)
plt.tight_layout()

fig3_path = FIG_DIR / "01_temporal_trace_lengths.png"
plt.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

## 9. Save artifacts

In [ ]:
results_path = ARTIFACT_DIR / "01_adaptime_reproduction_results.csv"
metrics_path = ARTIFACT_DIR / "01_adaptime_reproduction_metrics.csv"
summary_path = DOCS_DIR / "01_adaptime_reproduction_summary.md"

results.to_csv(results_path, index=False)
metrics.to_csv(metrics_path, index=False)

summary_md = f"""# Notebook 01 — AdapTime Reproduction Scaffold

Repo: `{REPO_NAME}`  
Notebook: `notebooks/01_adaptime_reproduction.ipynb`

## Purpose

This notebook implements a transparent AdapTime-style adaptive temporal reasoning baseline:

1. reformulate question intent
2. rewrite evidence into temporal facts / constraints
3. review answer against recovered trace

## Baseline metrics

| Metric | Value |
|---|---:|
| Exact-match accuracy | {accuracy:.3f} |
| Support rate | {support_rate:.3f} |
| Review pass rate | {pass_rate:.3f} |
| Number of cases | {len(results)} |

## Figures

- `figures/01_baseline_review_outcomes.png`
- `figures/01_adaptime_style_pipeline.png`
- `figures/01_temporal_trace_lengths.png`

## Handoff

Notebook 02 should add constraint-gated reasoning:

- encode stage outputs as vectors
- compute stage-to-stage cosine alignment
- flag reasoning drift
- introduce 45° phase-lock gate

Canonical gate:

```text
cos(theta) >= 1 / sqrt(1^2 + 1^2)
```
"""

summary_path.write_text(summary_md)

print(results_path)
print(metrics_path)
print(summary_path)

## 10. Phase-lock handoff preview

Notebook 02 can turn the baseline trace into a measurable system.

Proposed next additions:

```text
reformulate vector
rewrite vector
review vector
answer vector
```

Then evaluate stage alignment:

```text
cos(theta) >= 1 / sqrt(1^2 + 1^2)
```

Interpretation:

- above gate → temporal reasoning remains phase-locked
- below gate → reasoning drift / invalid assignment candidate
- residual review → identify which stage caused drift

In [ ]:
PHASE_LOCK_GATE = 1 / math.sqrt(1**2 + 1**2)
PHASE_LOCK_GATE

## 11. Final notebook status

In [ ]:
status = {
    "repo": REPO_NAME,
    "notebook": NOTEBOOK_ID,
    "cases": len(CASES),
    "accuracy": round(float(accuracy), 3),
    "support_rate": round(float(support_rate), 3),
    "review_pass_rate": round(float(pass_rate), 3),
    "phase_lock_gate_next": PHASE_LOCK_GATE,
    "artifacts": [
        str(results_path),
        str(metrics_path),
        str(summary_path),
        str(fig1_path),
        str(fig2_path),
        str(fig3_path),
    ],
}
print(json.dumps(status, indent=2))